# 01 - Data Quality and Schema Validation (Lab Support)

Use this notebook to validate cleaning logic before running containerized processing in AWS.

## Covers
- Raw partition inspection
- Local simulation of lab cleaning rules
- Row-drop diagnostics by partition
- Optional local Parquet preview output

In [ ]:
from pathlib import Path
import pandas as pd

if Path.cwd().name == 'notebooks':
    REPO_ROOT = Path.cwd().parent
else:
    REPO_ROOT = Path.cwd()

print('Repository root:', REPO_ROOT)

In [ ]:
raw_files = sorted((REPO_ROOT / 'data/raw').glob('trips_2023_*.csv'))
sample_file = REPO_ROOT / 'data' / 'sample' / 'sample_trips.csv'

print('Raw files:', [f.name for f in raw_files])
print('Sample file exists:', sample_file.exists())

In [ ]:
COLUMN_ALIASES = {
    'pickup_datetime': ['pickup_datetime', 'pickup_ts', 'pickup_time', 'tpep_pickup_datetime'],
    'pickup_zone_id': ['pickup_zone_id', 'pickup_zone', 'PULocationID'],
    'dropoff_zone_id': ['dropoff_zone_id', 'dropoff_zone', 'DOLocationID'],
    'trip_distance': ['trip_distance', 'distance_miles', 'distance'],
    'trip_duration_min': ['trip_duration_min', 'duration_min', 'duration_minutes'],
    'fare_amount': ['fare_amount', 'total_amount', 'fare'],
}


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {}
    for canonical, aliases in COLUMN_ALIASES.items():
        for alias in aliases:
            if alias in df.columns:
                rename_map[alias] = canonical
                break
    out = df.rename(columns=rename_map)
    required = list(COLUMN_ALIASES.keys())
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f'Missing required columns after normalization: {missing}')
    return out[required].copy()


def clean_like_lab(df: pd.DataFrame):
    x = normalize_columns(df)
    x['pickup_datetime'] = pd.to_datetime(x['pickup_datetime'], errors='coerce')

    numeric_cols = ['pickup_zone_id', 'dropoff_zone_id', 'trip_distance', 'trip_duration_min', 'fare_amount']
    for col in numeric_cols:
        x[col] = pd.to_numeric(x[col], errors='coerce')

    summary = {'input_rows': int(len(x))}

    null_mask = x.isnull().any(axis=1)
    summary['dropped_nulls'] = int(null_mask.sum())
    x = x.loc[~null_mask].copy()

    duration_mask = (x['trip_duration_min'] < 1.0) | (x['trip_duration_min'] > 180.0)
    summary['dropped_duration_outliers'] = int(duration_mask.sum())
    x = x.loc[~duration_mask].copy()

    distance_mask = (x['trip_distance'] < 0.1) | (x['trip_distance'] > 100.0)
    summary['dropped_distance_outliers'] = int(distance_mask.sum())
    x = x.loc[~distance_mask].copy()

    fare_mask = (x['fare_amount'] < 0.0) | (x['fare_amount'] > 500.0)
    summary['dropped_fare_outliers'] = int(fare_mask.sum())
    x = x.loc[~fare_mask].copy()

    x['pickup_zone_id'] = x['pickup_zone_id'].astype('int64')
    x['dropoff_zone_id'] = x['dropoff_zone_id'].astype('int64')
    x = x.sort_values('pickup_datetime').reset_index(drop=True)

    summary['output_rows'] = int(len(x))
    return x, summary

In [ ]:
summary_rows = []
for file in raw_files:
    raw_df = pd.read_csv(file)
    cleaned_df, summary = clean_like_lab(raw_df)
    summary['partition_file'] = file.name
    summary_rows.append(summary)

summary_df = pd.DataFrame(summary_rows)[[
    'partition_file', 'input_rows', 'dropped_nulls',
    'dropped_duration_outliers', 'dropped_distance_outliers',
    'dropped_fare_outliers', 'output_rows'
]]
summary_df

In [ ]:
sample_df = pd.read_csv(sample_file)
sample_cleaned, sample_summary = clean_like_lab(sample_df)
print(sample_summary)
sample_cleaned.head()

In [ ]:
out_dir = REPO_ROOT / 'notebooks' / 'output'
out_file = out_dir / 'cleaned_sample_preview.parquet'

try:
    sample_cleaned.to_parquet(out_file, index=False)
    print('Wrote:', out_file)
except Exception as exc:
    print('Parquet write skipped:', exc)

## Reflection Prompt
1. Which filter removes the highest proportion of rows?
2. How would one threshold change affect downstream analytics?
3. Why should this logic be containerized before EKS execution?